# LeetCode 399 — Evaluate Division bằng Python

Link đề: https://leetcode.com/problems/evaluate-division/ · Độ khó: **Medium** · Chủ đề: Graph, DFS, BFS, Union Find, Shortest Path, Floyd–Warshall

Skeleton LeetCode:

```python
class Solution:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        ...
```

Ghi chú: phần phát biểu đề dịch sang tiếng Việt từ đề gốc; mọi ví dụ được chạy lại và đối chiếu bằng code, kèm một **oracle độc lập** dựng từ "giá trị ẩn" của các biến.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các đẳng thức dạng `A / B = giá_trị` giữa các biến (chuỗi), hãy trả lời các truy vấn `C / D = ?`; nếu không suy ra được thì trả về `-1.0`.
2. Nắm công cụ Python: `dict` lồng nhau (đồ thị có trọng số), `defaultdict`, `set` đã thăm, đệ quy DFS, hàng đợi `deque` cho BFS, ba vòng lặp lồng của Floyd–Warshall, và mảng cha/trọng số của Union–Find.
3. Mô hình hóa bài toán thành **đồ thị**: mỗi biến là một đỉnh; đẳng thức `a / b = v` là cạnh `a → b` trọng số `v` và cạnh ngược `b → a` trọng số `1 / v`; đáp án một truy vấn là **tích các trọng số dọc theo một đường đi**.
4. Đi từ **bao đóng lặp đến khi không sinh thêm** → **DFS / BFS trên đồ thị** → **Floyd–Warshall nhân thay vì cộng** → **Union–Find có trọng số** (mỗi biến nhớ tỉ số của nó so với đại diện nhóm).
5. Tránh các lỗi kinh điển: quên cạnh ngược `1 / v`, quên kiểm tra biến chưa xuất hiện (`x / x` phải là `-1.0`), quên tập `visited` (đệ quy vô hạn), công thức gộp trọng số sai trong Union–Find, cộng thay vì nhân.

## 2. Đề bài & ví dụ

Cho mảng các cặp biến `equations` và mảng số thực `values`, trong đó `equations[i] = [A_i, B_i]` và `values[i]` nghĩa là `A_i / B_i = values[i]`. Mỗi biến là một chuỗi (chữ thường và chữ số, dài `1..5`).

Cho thêm các truy vấn `queries[j] = [C_j, D_j]`: hãy tính `C_j / D_j`. Nếu **không thể xác định** đáp án thì trả về `-1.0`.

Lưu ý: đầu vào luôn hợp lệ (**không mâu thuẫn**, không chia cho 0). Biến **không xuất hiện** trong `equations` là **chưa xác định** nên mọi truy vấn chứa nó đều trả `-1.0` (kể cả `x / x`).

**Ví dụ 1:** `equations = [["a","b"],["b","c"]]`, `values = [2.0, 3.0]`, `queries = [["a","c"],["b","a"],["a","e"],["a","a"],["x","x"]]` → `[6.0, 0.5, -1.0, 1.0, -1.0]`.

**Ví dụ 2:** `equations = [["a","b"],["b","c"],["bc","cd"]]`, `values = [1.5, 2.5, 5.0]`, `queries = [["a","c"],["c","b"],["bc","cd"],["cd","bc"]]` → `[3.75, 0.4, 5.0, 0.2]`.

**Ví dụ 3:** `equations = [["a","b"]]`, `values = [0.5]`, `queries = [["a","b"],["b","a"],["a","c"],["x","y"]]` → `[0.5, 2.0, -1.0, -1.0]`.

**Ràng buộc:** `1 <= equations.length <= 20`; `equations[i].length == 2`; `1 <= len(A_i), len(B_i) <= 5`; `values.length == equations.length`; `0.0 < values[i] <= 20.0`; `1 <= queries.length <= 20`; các biến gồm chữ thường và chữ số.

In [1]:
from collections import defaultdict, deque
from fractions import Fraction
import random
import time
from typing import List

examples = [
    ([["a", "b"], ["b", "c"]], [2.0, 3.0],
     [["a", "c"], ["b", "a"], ["a", "e"], ["a", "a"], ["x", "x"]],
     [6.0, 0.5, -1.0, 1.0, -1.0]),
    ([["a", "b"], ["b", "c"], ["bc", "cd"]], [1.5, 2.5, 5.0],
     [["a", "c"], ["c", "b"], ["bc", "cd"], ["cd", "bc"]],
     [3.75, 0.4, 5.0, 0.2]),
    ([["a", "b"]], [0.5],
     [["a", "b"], ["b", "a"], ["a", "c"], ["x", "y"]],
     [0.5, 2.0, -1.0, -1.0]),
]


def same_answers(got, expected, tol=1e-5):
    """So khớp danh sách số thực theo sai số tuyệt đối 1e-5 (như bộ chấm LeetCode); -1.0 phải khớp đúng."""
    if len(got) != len(expected):
        return False
    for g, e in zip(got, expected):
        if e == -1.0:
            if g != -1.0:
                return False
        elif abs(g - e) > tol:
            return False
    return True


for eq, val, qs, exp in examples:
    print(eq, val, "| số truy vấn:", len(qs), "-> kỳ vọng", exp)

[['a', 'b'], ['b', 'c']] [2.0, 3.0] | số truy vấn: 5 -> kỳ vọng [6.0, 0.5, -1.0, 1.0, -1.0]
[['a', 'b'], ['b', 'c'], ['bc', 'cd']] [1.5, 2.5, 5.0] | số truy vấn: 4 -> kỳ vọng [3.75, 0.4, 5.0, 0.2]
[['a', 'b']] [0.5] | số truy vấn: 4 -> kỳ vọng [0.5, 2.0, -1.0, -1.0]


## 3. Kiến thức nền

### 3.1. Số thực và so sánh có sai số

Phép chia/nhân số thực (`float`) có sai số làm tròn: `0.1 * 3` không đúng bằng `0.3`. Vì vậy ta **không so `==`** mà so `abs(a - b) <= 1e-5`. (Chính bộ chấm của LeetCode cũng cho phép sai số `1e-5`.)

In [2]:
print(0.1 * 3, 0.1 * 3 == 0.3, abs(0.1 * 3 - 0.3) < 1e-9)
print(1 / 3 * 3, 2.5 * 1.5, 1 / 2.5)

0.30000000000000004 False True
1.0 3.75 0.4


### 3.2. Đồ thị bằng `dict` lồng nhau

Ta lưu đồ thị có trọng số bằng `graph[u][v] = trọng số cạnh u → v`. `defaultdict(dict)` tự tạo `dict` rỗng cho đỉnh mới. Kiểm tra cạnh/đỉnh bằng toán tử `in`.

In [3]:
graph = defaultdict(dict)
graph["a"]["b"] = 2.0
graph["b"]["a"] = 1 / 2.0
graph["b"]["c"] = 3.0
graph["c"]["b"] = 1 / 3.0
print(dict(graph))
print("a có trong đồ thị?", "a" in graph, "| x có trong đồ thị?", "x" in graph)
print("cạnh a->b:", graph["a"]["b"], "| các láng giềng của b:", list(graph["b"].items()))

{'a': {'b': 2.0}, 'b': {'a': 0.5, 'c': 3.0}, 'c': {'b': 0.3333333333333333}}
a có trong đồ thị? True | x có trong đồ thị? False
cạnh a->b: 2.0 | các láng giềng của b: [('a', 0.5), ('c', 3.0)]


### 3.3. Duyệt đồ thị: DFS (đệ quy) và BFS (hàng đợi)

- **DFS** (depth-first): đi sâu theo một nhánh, hết đường mới quay lui; cài bằng đệ quy. Cần tập `visited` để **không đi vòng lặp**.
- **BFS** (breadth-first): thăm theo "tầng"; cài bằng `deque` (`append` thêm phải, `popleft` lấy trái).

Cả hai đều thăm mỗi đỉnh tối đa một lần: `O(V + E)`.

In [4]:
g = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}


def dfs_order(u, seen):
    seen.add(u)
    order = [u]
    for v in g[u]:
        if v not in seen:
            order += dfs_order(v, seen)
    return order


def bfs_order(start):
    seen, q, order = {start}, deque([start]), []
    while q:
        u = q.popleft()
        order.append(u)
        for v in g[u]:
            if v not in seen:
                seen.add(v)
                q.append(v)
    return order


print("DFS:", dfs_order("a", set()))      # a b d c
print("BFS:", bfs_order("a"))             # a b c d

DFS: ['a', 'b', 'd', 'c']
BFS: ['a', 'b', 'c', 'd']


### 3.4. Floyd–Warshall: mọi cặp đỉnh

Ý tưởng: thử cho **mọi đỉnh trung gian `k`**; nếu có đường `i → k` và `k → j` thì có đường `i → j`. Ba vòng `for k, for i, for j`, `O(V^3)`. Bản quen thuộc **cộng** độ dài; ở bài này ta **nhân** các tỉ số. Thứ tự vòng ngoài cùng là `k` (trung gian) — **bắt buộc**.

In [5]:
# Ví dụ nhỏ: đường đi có hay không (reachability) bằng Floyd–Warshall cùng khung 3 vòng lặp
nodes = ["a", "b", "c", "d"]
idx = {v: i for i, v in enumerate(nodes)}
reach = [[i == j for j in range(4)] for i in range(4)]
for u, v in [("a", "b"), ("b", "c")]:
    reach[idx[u]][idx[v]] = True
for k in range(4):
    for i in range(4):
        for j in range(4):
            if reach[i][k] and reach[k][j]:
                reach[i][j] = True
print("a tới c được không?", reach[idx["a"]][idx["c"]], "| a tới d được không?", reach[idx["a"]][idx["d"]])

a tới c được không? True | a tới d được không? False


### 3.5. Union–Find (tập hợp rời nhau) và "trọng số so với gốc"

**Union–Find** quản lý các nhóm: `parent[x]` trỏ lên cha, gốc của nhóm trỏ chính nó; `find(x)` tìm gốc (có **nén đường đi** để lần sau nhanh); `union` gộp hai nhóm. Ở bài này mỗi biến còn nhớ thêm `weight[x] = giá trị của x / giá trị của gốc nhóm`; khi nén đường đi ta **nhân dồn** các trọng số trên đường. Ta sẽ xây trong Cách 5; phiên bản thường (không trọng số) dưới đây để quen.

In [6]:
parent = {x: x for x in "abcde"}


def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])          # nén đường đi
    return parent[x]


def union(x, y):
    parent[find(x)] = find(y)


union("a", "b")
union("b", "c")
print({x: find(x) for x in "abcde"})          # a, b, c cùng một gốc; d, e riêng
print("a và c cùng nhóm?", find("a") == find("c"), "| a và d cùng nhóm?", find("a") == find("d"))

{'a': 'c', 'b': 'c', 'c': 'c', 'd': 'd', 'e': 'e'}
a và c cùng nhóm? True | a và d cùng nhóm? False


## 4. Ý tưởng nền tảng: tính tay

Cho `a / b = 2` và `b / c = 3`.

- `a / c = (a / b) * (b / c) = 2 * 3 = 6`: **nhân** hai tỉ số "nối đuôi" nhau (`b` triệt tiêu).
- `b / a = 1 / (a / b) = 1/2 = 0.5`: **đảo ngược** tỉ số.
- `a / a = 1` (nếu `a` đã xuất hiện). `a / e`: không có `e` trong dữ kiện nên **không xác định** `-1.0`.
- `x / x`: `x` chưa xuất hiện ⇒ chưa xác định ⇒ `-1.0` (đề nhấn mạnh điểm này).

Vậy: biến = đỉnh, `a / b = v` = cạnh `a → b` trọng số `v` cùng cạnh `b → a` trọng số `1 / v`; `C / D` = **tích trọng số dọc theo một đường đi** từ `C` tới `D`. (Do đầu vào không mâu thuẫn, **mọi đường đi** cho cùng một tích.) Nếu không có đường đi thì đáp án `-1.0`.

## 5. Cách 1 — Brute Force: mở rộng bao đóng cho tới khi không sinh thêm

**Ý tưởng:** giữ một bảng `known[(x, y)] = x / y` các tỉ số đã biết. Ban đầu có các đẳng thức đề cho, cạnh ngược `1 / v` và `x / x = 1` cho mọi biến. Lặp: với mọi cặp đã biết `(x, y)` và `(y, z)`, suy ra `(x, z) = known[(x, y)] * known[(y, z)]`. Dừng khi một vòng lặp **không thêm gì mới**. Truy vấn `C / D` tra thẳng bảng.

In [7]:
def calc_closure(equations, values, queries):
    known = {}
    variables = set()
    for (a, b), v in zip(equations, values):
        variables.update((a, b))
        known[(a, b)] = v
        known[(b, a)] = 1.0 / v
    for x in variables:
        known[(x, x)] = 1.0
    changed = True
    while changed:                                   # lặp tới khi bảng không phình thêm
        changed = False
        for (x, y), v1 in list(known.items()):
            for (y2, z), v2 in list(known.items()):
                if y == y2 and (x, z) not in known:
                    known[(x, z)] = v1 * v2
                    changed = True
    return [known.get((c, d), -1.0) for c, d in queries]


for eq, val, qs, exp in examples:
    got = calc_closure(eq, val, qs)
    print(got, "| kỳ vọng", exp)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0] | kỳ vọng [6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2] | kỳ vọng [3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0] | kỳ vọng [0.5, 2.0, -1.0, -1.0]


**Độ phức tạp:** gọi `V` là số biến. Bảng tối đa `V^2` cặp; mỗi vòng duyệt `O((V^2)^2) = O(V^4)` cặp-cặp, và có thể cần tới `V` vòng. Với `V <= 40` vẫn chạy được, nhưng rất lãng phí. Space `O(V^2)`.

**Nút thắt:** mỗi vòng ta **duyệt lại tất cả cặp** dù hầu hết đã xử lý. Cần một cách có thứ tự hơn: xét một truy vấn là **hỏi có đường đi không** — hãy dùng đồ thị.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá 1 — nhìn như đồ thị:** biến là đỉnh, đẳng thức là cạnh hai chiều với trọng số `v` và `1/v`. Một truy vấn `C / D` chỉ cần **một đường đi** từ `C` tới `D`, đáp án là tích trọng số trên đường đó. Không cần tính mọi cặp.

**Bước đột phá 2 — tính tất cả bằng một "đại diện nhóm":** vì đầu vào không mâu thuẫn, trong mỗi **thành phần liên thông**, ta có thể chọn một biến làm gốc và biểu diễn mọi biến theo gốc: `x = r_x * gốc` (giá trị thật không biết, nhưng **tỉ số** thì biết). Khi đó `C / D = r_C / r_D`, chỉ cần **hai tra cứu** thay vì tìm đường. Đó chính là **Union–Find có trọng số**.

Hãy kiểm chứng ý tưởng 2 bằng một ví dụ: gốc `c`, `b = 3c` (vì `b / c = 3`), `a = 2b = 6c`. Khi đó `a / b = 6/3 = 2` ✓, `b / a = 3/6 = 0.5` ✓.

In [8]:
# Mỗi biến biểu diễn theo gốc c: r[x] = x / c
r = {"c": 1.0}
r["b"] = 3.0 * r["c"]              # b / c = 3
r["a"] = 2.0 * r["b"]              # a / b = 2  =>  a = 2b
print(r)
print("a/b =", r["a"] / r["b"], "| b/a =", r["b"] / r["a"], "| a/c =", r["a"] / r["c"])

{'c': 1.0, 'b': 3.0, 'a': 6.0}
a/b = 2.0 | b/a = 0.5 | a/c = 6.0


## 7. Các cách tiếp theo

### Cách 2 — DFS trên đồ thị (từng truy vấn)

**Giải thích từng đoạn:**

1. Dựng `graph[a][b] = v` và `graph[b][a] = 1 / v` cho mỗi đẳng thức.
2. Với mỗi truy vấn `(c, d)`: nếu `c` hoặc `d` **không có** trong `graph` ⇒ `-1.0`; nếu `c == d` ⇒ `1.0`.
3. Ngược lại DFS từ `c` mang theo "tích hiện tại" `acc`; tới `d` thì trả `acc`. Dùng `visited` để không đi vòng; hết đường không tới được thì `-1.0`.

In [9]:
def build_graph(equations, values):
    graph = defaultdict(dict)
    for (a, b), v in zip(equations, values):
        graph[a][b] = v
        graph[b][a] = 1.0 / v
    return graph


def calc_dfs(equations, values, queries):
    graph = build_graph(equations, values)

    def dfs(u, target, acc, visited):
        if u == target:
            return acc
        visited.add(u)
        for nxt, w in graph[u].items():
            if nxt not in visited:
                res = dfs(nxt, target, acc * w, visited)
                if res != -1.0:
                    return res
        return -1.0

    ans = []
    for c, d in queries:
        if c not in graph or d not in graph:
            ans.append(-1.0)
        else:
            ans.append(dfs(c, d, 1.0, set()))
    return ans


for eq, val, qs, exp in examples:
    got = calc_dfs(eq, val, qs)
    print(got)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0]


In [10]:
# Code nộp LeetCode — Cách 2 (DFS)
class SolutionDFS:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        graph = defaultdict(dict)
        for (a, b), v in zip(equations, values):
            graph[a][b] = v
            graph[b][a] = 1.0 / v

        def dfs(u: str, target: str, acc: float, visited: set) -> float:
            if u == target:
                return acc
            visited.add(u)
            for nxt, w in graph[u].items():
                if nxt not in visited:
                    res = dfs(nxt, target, acc * w, visited)
                    if res != -1.0:
                        return res
            return -1.0

        return [dfs(c, d, 1.0, set()) if c in graph and d in graph else -1.0 for c, d in queries]


print(SolutionDFS().calcEquation(*examples[0][:3]))

[6.0, 0.5, -1.0, 1.0, -1.0]


**Độ phức tạp Cách 2:** mỗi truy vấn `O(V + E)`; tổng `O(Q * (V + E))` với `Q` truy vấn. Space `O(V + E)` (đồ thị) và độ sâu đệ quy `O(V)`.

### Cách 3 — BFS trên đồ thị

**Giải thích từng đoạn:** cùng đồ thị; thay đệ quy bằng hàng đợi chứa cặp `(đỉnh, tích từ nguồn tới đỉnh đó)`. Lần đầu chạm `d` ta có ngay đáp án (đường ngắn nhất theo số cạnh — vì mọi đường cho cùng tích, ta không cần xét đường khác). Ưu điểm: **không đệ quy**, không lo tràn ngăn xếp.

In [11]:
def calc_bfs(equations, values, queries):
    graph = build_graph(equations, values)
    ans = []
    for c, d in queries:
        if c not in graph or d not in graph:
            ans.append(-1.0)
            continue
        found = -1.0
        queue = deque([(c, 1.0)])
        seen = {c}
        while queue:
            u, acc = queue.popleft()
            if u == d:
                found = acc
                break
            for nxt, w in graph[u].items():
                if nxt not in seen:
                    seen.add(nxt)
                    queue.append((nxt, acc * w))
        ans.append(found)
    return ans


for eq, val, qs, exp in examples:
    got = calc_bfs(eq, val, qs)
    print(got)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0]


In [12]:
# Code nộp LeetCode — Cách 3 (BFS)
class SolutionBFS:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        graph = defaultdict(dict)
        for (a, b), v in zip(equations, values):
            graph[a][b] = v
            graph[b][a] = 1.0 / v
        ans = []
        for c, d in queries:
            if c not in graph or d not in graph:
                ans.append(-1.0)
                continue
            res = -1.0
            queue = deque([(c, 1.0)])
            seen = {c}
            while queue:
                u, acc = queue.popleft()
                if u == d:
                    res = acc
                    break
                for nxt, w in graph[u].items():
                    if nxt not in seen:
                        seen.add(nxt)
                        queue.append((nxt, acc * w))
            ans.append(res)
        return ans


print(SolutionBFS().calcEquation(*examples[1][:3]))

[3.75, 0.4, 5.0, 0.2]


**Độ phức tạp Cách 3:** như Cách 2: `O(Q * (V + E))`; Space `O(V + E)`.

### Cách 4 — Floyd–Warshall (tính trước mọi cặp)

**Giải thích từng đoạn:**

1. Đánh số các biến `0..V-1`. Ma trận `g[i][j]` ban đầu `-1.0` (chưa biết); `g[i][i] = 1.0`; mỗi đẳng thức `a / b = v` đặt `g[a][b] = v`, `g[b][a] = 1 / v`.
2. Ba vòng `k, i, j`: nếu `g[i][k]` và `g[k][j]` đã biết mà `g[i][j]` chưa biết thì `g[i][j] = g[i][k] * g[k][j]`.
3. Mỗi truy vấn tra ma trận trong `O(1)`; biến chưa xuất hiện ⇒ `-1.0`.

In [13]:
def calc_floyd(equations, values, queries):
    ids = {}
    for a, b in equations:
        for x in (a, b):
            if x not in ids:
                ids[x] = len(ids)
    m = len(ids)
    g = [[-1.0] * m for _ in range(m)]
    for i in range(m):
        g[i][i] = 1.0
    for (a, b), v in zip(equations, values):
        g[ids[a]][ids[b]] = v
        g[ids[b]][ids[a]] = 1.0 / v
    for k in range(m):                       # k là đỉnh trung gian: PHẢI là vòng ngoài cùng
        for i in range(m):
            for j in range(m):
                if g[i][j] < 0 and g[i][k] > 0 and g[k][j] > 0:
                    g[i][j] = g[i][k] * g[k][j]
    ans = []
    for c, d in queries:
        if c in ids and d in ids:
            ans.append(g[ids[c]][ids[d]])
        else:
            ans.append(-1.0)
    return ans


for eq, val, qs, exp in examples:
    got = calc_floyd(eq, val, qs)
    print(got)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0]


In [14]:
# Code nộp LeetCode — Cách 4 (Floyd–Warshall)
class SolutionFloyd:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        ids = {}
        for a, b in equations:
            for x in (a, b):
                if x not in ids:
                    ids[x] = len(ids)
        m = len(ids)
        g = [[-1.0] * m for _ in range(m)]
        for i in range(m):
            g[i][i] = 1.0
        for (a, b), v in zip(equations, values):
            g[ids[a]][ids[b]] = v
            g[ids[b]][ids[a]] = 1.0 / v
        for k in range(m):
            for i in range(m):
                for j in range(m):
                    if g[i][j] < 0 and g[i][k] > 0 and g[k][j] > 0:
                        g[i][j] = g[i][k] * g[k][j]
        return [g[ids[c]][ids[d]] if c in ids and d in ids else -1.0 for c, d in queries]


print(SolutionFloyd().calcEquation(*examples[2][:3]))

[0.5, 2.0, -1.0, -1.0]


**Độ phức tạp Cách 4:** Time `O(V^3 + Q)`, Space `O(V^2)`. Tốt khi **nhiều truy vấn** và `V` nhỏ (ở đề: `V <= 40`).

### Cách 5 — Union–Find có trọng số (chốt)

**Giải thích từng đoạn:**

1. `parent[x]` là cha; `weight[x] = x / parent[x]` (tỉ số của `x` so với **cha**); ban đầu `parent[x] = x`, `weight[x] = 1.0`.
2. `find(x)` trả gốc và **nén đường đi**: khi gắn `x` thẳng vào gốc, `weight[x]` nhân thêm `weight[cha_cũ]` (vì `x/gốc = (x/cha) * (cha/gốc)`).
3. Gộp đẳng thức `a / b = v`: `ra = find(a)`, `rb = find(b)` (sau đó `weight[a] = a/ra`, `weight[b] = b/rb`). Nếu khác gốc, gắn `ra` dưới `rb`: `parent[ra] = rb` và `weight[ra] = ra / rb`. Từ `a/b = v` ta có `ra/rb = (a/b) * (b/rb) / (a/ra) = v * weight[b] / weight[a]`.
4. Truy vấn `c / d`: nếu thiếu biến hoặc `find(c) != find(d)` ⇒ `-1.0`; ngược lại `weight[c] / weight[d]` (cùng chia cho gốc nên triệt tiêu).

In [15]:
def calc_union_find(equations, values, queries):
    parent, weight = {}, {}

    def find(x):
        if parent[x] != x:
            origin = parent[x]
            parent[x] = find(origin)                   # sau lệnh này origin đã trỏ thẳng tới gốc
            weight[x] *= weight[origin]                # x/gốc = (x/origin) * (origin/gốc)
        return parent[x]

    for a, b in equations:
        for x in (a, b):
            if x not in parent:
                parent[x], weight[x] = x, 1.0

    for (a, b), v in zip(equations, values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            weight[ra] = v * weight[b] / weight[a]     # ra/rb = (a/b) * (b/rb) / (a/ra)

    ans = []
    for c, d in queries:
        if c in parent and d in parent and find(c) == find(d):
            ans.append(weight[c] / weight[d])
        else:
            ans.append(-1.0)
    return ans


for eq, val, qs, exp in examples:
    got = calc_union_find(eq, val, qs)
    print(got)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0]


In [16]:
# Code nộp LeetCode — Cách 5 (Union–Find có trọng số) — cũng là lời giải chính (xem mục 9)
class SolutionUF:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        parent = {}
        weight = {}

        def find(x: str) -> str:
            if parent[x] != x:
                origin = parent[x]
                parent[x] = find(origin)
                weight[x] *= weight[origin]
            return parent[x]

        for a, b in equations:
            for x in (a, b):
                if x not in parent:
                    parent[x], weight[x] = x, 1.0
        for (a, b), v in zip(equations, values):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                weight[ra] = v * weight[b] / weight[a]
        return [weight[c] / weight[d] if c in parent and d in parent and find(c) == find(d) else -1.0
                for c, d in queries]


print(SolutionUF().calcEquation(*examples[0][:3]))

[6.0, 0.5, -1.0, 1.0, -1.0]


**Độ phức tạp Cách 5:** Time `O((E + Q) * α(V))` (`α` là hàm Ackermann ngược, coi như hằng) — mỗi truy vấn chỉ hai lần `find`; Space `O(V)`.

## 8. TRACE từng bước

**Trace DFS** cho `a / c` ở ví dụ 1 (đồ thị `a→b: 2`, `b→a: 0.5`, `b→c: 3`, `c→b: 1/3`):

| bước | đỉnh hiện tại `u` | tích `acc` | cạnh đi tiếp | ghi chú |
|---|---|---|---|---|
| 1 | `a` | 1 | `a → b` (trọng số 2) | `visited = {a}` |
| 2 | `b` | 2 | `b → a` bỏ (đã thăm), `b → c` (trọng số 3) | `visited = {a, b}` |
| 3 | `c` | 6 | — | `u == target` ⇒ trả `6` |

**Trace Union–Find** (cùng ví dụ): gộp `a/b = 2` rồi `b/c = 3`:

| thao tác | `parent` | `weight` (so với cha) |
|---|---|---|
| khởi tạo | `a→a, b→b, c→c` | `a=1, b=1, c=1` |
| `a/b=2`: `ra=a, rb=b` | `a→b` | `a = 2·1/1 = 2` |
| `b/c=3`: `ra=b, rb=c` | `b→c` | `b = 3` |
| truy vấn `a/c`: `find(a)` nén | `a→c` | `a = 2·3 = 6` |

Hai hàm `trace_dfs` và `trace_union_find` dưới đây in đúng các cột đó.

In [17]:
def trace_dfs(equations, values, c, d):
    graph = build_graph(equations, values)
    print(f"đồ thị: { {u: dict(vs) for u, vs in graph.items()} }")
    print(f"truy vấn {c} / {d}")
    if c not in graph or d not in graph:
        print("thiếu biến -> -1.0")
        return -1.0
    path = []

    def dfs(u, acc, visited):
        path.append(u)
        print(f"{'  ' * (len(path) - 1)}thăm {u}, acc = {acc:.5g}, visited = {sorted(visited | {u})}")
        if u == d:
            print(f"{'  ' * (len(path) - 1)}=> chạm {d}, trả {acc:.5g}")
            return acc
        for nxt, w in graph[u].items():
            if nxt in visited | {u}:
                print(f"{'  ' * len(path)}bỏ {u}->{nxt} (đã thăm)")
                continue
            print(f"{'  ' * len(path)}đi {u}->{nxt} (nhân {w:.5g})")
            res = dfs(nxt, acc * w, visited | {u})
            if res != -1.0:
                return res
        path.pop()
        return -1.0

    return dfs(c, 1.0, set())


print("kết quả:", trace_dfs([["a", "b"], ["b", "c"]], [2.0, 3.0], "a", "c"))

đồ thị: {'a': {'b': 2.0}, 'b': {'a': 0.5, 'c': 3.0}, 'c': {'b': 0.3333333333333333}}
truy vấn a / c
thăm a, acc = 1, visited = ['a']
  đi a->b (nhân 2)
  thăm b, acc = 2, visited = ['a', 'b']
    bỏ b->a (đã thăm)
    đi b->c (nhân 3)
    thăm c, acc = 6, visited = ['a', 'b', 'c']
    => chạm c, trả 6
kết quả: 6.0


In [18]:
def trace_union_find(equations, values, queries):
    parent, weight = {}, {}

    def show(title):
        print(f"{title:<26} parent={parent}  weight={ {k: round(w, 4) for k, w in weight.items()} }")

    def find(x):
        if parent[x] != x:
            origin = parent[x]
            parent[x] = find(origin)
            weight[x] *= weight[origin]
        return parent[x]

    for a, b in equations:
        for x in (a, b):
            if x not in parent:
                parent[x], weight[x] = x, 1.0
    show("khởi tạo")
    for (a, b), v in zip(equations, values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            weight[ra] = v * weight[b] / weight[a]
            show(f"gộp {a}/{b}={v}: {ra}->{rb}")
        else:
            show(f"{a}/{b}={v}: cùng nhóm")
    for c, d in queries:
        if c in parent and d in parent and find(c) == find(d):
            show(f"truy vấn {c}/{d}")
            print(f"   = weight[{c}] / weight[{d}] = {weight[c]:.5g} / {weight[d]:.5g} = {weight[c] / weight[d]:.5g}")
        else:
            print(f"truy vấn {c}/{d}: thiếu biến hoặc khác nhóm -> -1.0")


trace_union_find([["a", "b"], ["b", "c"]], [2.0, 3.0], [["a", "c"], ["b", "a"], ["a", "e"]])

khởi tạo                   parent={'a': 'a', 'b': 'b', 'c': 'c'}  weight={'a': 1.0, 'b': 1.0, 'c': 1.0}
gộp a/b=2.0: a->b          parent={'a': 'b', 'b': 'b', 'c': 'c'}  weight={'a': 2.0, 'b': 1.0, 'c': 1.0}
gộp b/c=3.0: b->c          parent={'a': 'b', 'b': 'c', 'c': 'c'}  weight={'a': 2.0, 'b': 3.0, 'c': 1.0}
truy vấn a/c               parent={'a': 'c', 'b': 'c', 'c': 'c'}  weight={'a': 6.0, 'b': 3.0, 'c': 1.0}
   = weight[a] / weight[c] = 6 / 1 = 6
truy vấn b/a               parent={'a': 'c', 'b': 'c', 'c': 'c'}  weight={'a': 6.0, 'b': 3.0, 'c': 1.0}
   = weight[b] / weight[a] = 3 / 6 = 0.5
truy vấn a/e: thiếu biến hoặc khác nhóm -> -1.0


In [19]:
# Trace thứ hai với ví dụ 2 (chuỗi a-b-c và cặp rời bc-cd)
trace_union_find([["a", "b"], ["b", "c"], ["bc", "cd"]], [1.5, 2.5, 5.0], [["a", "c"], ["c", "b"], ["cd", "bc"], ["a", "bc"]])

khởi tạo                   parent={'a': 'a', 'b': 'b', 'c': 'c', 'bc': 'bc', 'cd': 'cd'}  weight={'a': 1.0, 'b': 1.0, 'c': 1.0, 'bc': 1.0, 'cd': 1.0}
gộp a/b=1.5: a->b          parent={'a': 'b', 'b': 'b', 'c': 'c', 'bc': 'bc', 'cd': 'cd'}  weight={'a': 1.5, 'b': 1.0, 'c': 1.0, 'bc': 1.0, 'cd': 1.0}
gộp b/c=2.5: b->c          parent={'a': 'b', 'b': 'c', 'c': 'c', 'bc': 'bc', 'cd': 'cd'}  weight={'a': 1.5, 'b': 2.5, 'c': 1.0, 'bc': 1.0, 'cd': 1.0}
gộp bc/cd=5.0: bc->cd      parent={'a': 'b', 'b': 'c', 'c': 'c', 'bc': 'cd', 'cd': 'cd'}  weight={'a': 1.5, 'b': 2.5, 'c': 1.0, 'bc': 5.0, 'cd': 1.0}
truy vấn a/c               parent={'a': 'c', 'b': 'c', 'c': 'c', 'bc': 'cd', 'cd': 'cd'}  weight={'a': 3.75, 'b': 2.5, 'c': 1.0, 'bc': 5.0, 'cd': 1.0}
   = weight[a] / weight[c] = 3.75 / 1 = 3.75
truy vấn c/b               parent={'a': 'c', 'b': 'c', 'c': 'c', 'bc': 'cd', 'cd': 'cd'}  weight={'a': 3.75, 'b': 2.5, 'c': 1.0, 'bc': 5.0, 'cd': 1.0}
   = weight[c] / weight[b] = 1 / 2.5 = 0.4
truy vấn c

## 9. Code nộp LeetCode

Lời giải chính của notebook là **Cách 5** (Union–Find có trọng số): mỗi truy vấn chỉ hai lần `find`, không đệ quy sâu trên đồ thị, dễ mở rộng thêm đẳng thức.

In [20]:
class Solution:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        parent = {}       # parent[x]: cha của x
        weight = {}       # weight[x]: x / parent[x]

        def find(x: str) -> str:
            if parent[x] != x:
                origin = parent[x]
                parent[x] = find(origin)           # nén đường đi
                weight[x] *= weight[origin]        # x/gốc = (x/origin) * (origin/gốc)
            return parent[x]

        for a, b in equations:
            for x in (a, b):
                if x not in parent:
                    parent[x], weight[x] = x, 1.0

        for (a, b), v in zip(equations, values):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                weight[ra] = v * weight[b] / weight[a]

        ans = []
        for c, d in queries:
            if c in parent and d in parent and find(c) == find(d):
                ans.append(weight[c] / weight[d])
            else:
                ans.append(-1.0)
        return ans


for eq, val, qs, exp in examples:
    got = Solution().calcEquation(eq, val, qs)
    print(got)
    assert same_answers(got, exp)

[6.0, 0.5, -1.0, 1.0, -1.0]
[3.75, 0.4, 5.0, 0.2]
[0.5, 2.0, -1.0, -1.0]


## 10. Vì sao đúng

1. **Mô hình:** nếu `a / b = v` thì `b / a = 1 / v`; nếu `a / b = p` và `b / c = q` thì `a / c = p * q`. Mọi tỉ số suy ra được đều là tích dọc theo một đường đi trong đồ thị cạnh hai chiều. Ngược lại, hai biến khác thành phần liên thông **không có ràng buộc** nên không xác định (`-1.0`).
2. **Không mâu thuẫn ⇒ đường nào cũng cùng kết quả:** đầu vào luôn hợp lệ nên ta có thể dùng đường đi bất kỳ (DFS/BFS tìm đường nào cũng được).
3. **Union–Find:** bất biến sau mỗi `find(x)`: `weight[x] = x / parent[x]` và `parent[x]` là gốc. Công thức gộp: `ra / rb = (ra/a) * (a/b) * (b/rb) = v * weight[b] / weight[a]` ✓. Truy vấn: `c/d = (c/gốc) / (d/gốc) = weight[c] / weight[d]` ✓.
4. **Biến chưa xuất hiện:** không có trong `parent` (hoặc đồ thị) nên bị loại bằng điều kiện `c in parent`; nhờ đó `x / x = -1.0` đúng đề.
5. **Floyd–Warshall:** quy nạp theo `k`: sau khi xét trung gian `0..k`, `g[i][j]` đã biết nếu có đường đi chỉ qua các trung gian trong `0..k`.

## 11. Time / Space Complexity

Gọi `V` là số biến khác nhau (`<= 2 * 20 = 40`), `E` số đẳng thức, `Q` số truy vấn.

| Cách | Time | Space |
|---|---|---|
| 1. Bao đóng | `O(V^4 ... V^5)` | `O(V^2)` |
| 2. DFS | `O(Q * (V + E))` | `O(V + E)` |
| 3. BFS | `O(Q * (V + E))` | `O(V + E)` |
| 4. Floyd–Warshall | `O(V^3 + Q)` | `O(V^2)` |
| 5. Union–Find có trọng số | `O((E + Q) * α(V))` | `O(V)` |

Với ràng buộc nhỏ của đề, mọi cách (trừ có thể Cách 1) đều chạy tức thì; sự khác biệt lộ ra khi `V`, `Q` lớn hơn nhiều (bài tương tự ngoài LeetCode).

## 12. Bộ test

Oracle độc lập: **giá trị ẩn**. Sinh ngẫu nhiên (seed cố định) một giá trị hữu tỉ ẩn `h[x]` cho mỗi biến, rồi lấy đẳng thức `a / b = h[a] / h[b]` giữa các cặp ngẫu nhiên. Đáp án thật của truy vấn là `h[c] / h[d]` **nếu** `c`, `d` ở cùng thành phần liên thông (tính bằng gộp tập đơn giản, không dùng Union–Find), ngược lại `-1.0`. Đầu vào sinh ra luôn đúng ràng buộc (`values` trong `(0, 20]`).

In [21]:
ALL = [calc_closure, calc_dfs, calc_bfs, calc_floyd, calc_union_find]
CLASSES = [Solution(), SolutionDFS(), SolutionBFS(), SolutionFloyd(), SolutionUF()]

# 12.1 Ví dụ của đề cho cả 5 hàm và 5 lớp
for eq, val, qs, exp in examples:
    for f in ALL:
        assert same_answers(f(eq, val, qs), exp), f.__name__
    for sol in CLASSES:
        assert same_answers(sol.calcEquation(eq, val, qs), exp), type(sol).__name__
print("OK: ví dụ đề cho 5 hàm demo và 5 lớp nộp")

OK: ví dụ đề cho 5 hàm demo và 5 lớp nộp


In [22]:
# 12.2 Biên: phương trình tự quy chiếu, một biến, chuỗi dài, truy vấn trùng, biến không xuất hiện
edge_cases = [
    ([["a", "a"]], [1.0], [["a", "a"], ["a", "b"], ["b", "b"]], [1.0, -1.0, -1.0]),
    ([["a", "b"], ["b", "a"]], [2.0, 0.5], [["a", "b"], ["b", "a"]], [2.0, 0.5]),          # lặp ngược nhưng nhất quán
    ([["a", "b"], ["c", "d"]], [2.0, 3.0], [["a", "c"], ["a", "d"], ["d", "c"], ["c", "d"]], [-1.0, -1.0, 1 / 3, 3.0]),
    ([["x1", "y2"]], [20.0], [["x1", "y2"], ["y2", "x1"], ["x1", "x1"], ["zzzzz", "x1"]], [20.0, 0.05, 1.0, -1.0]),
]
# chuỗi dài 20 đẳng thức v0/v1 = 2, v1/v2 = 2, ...
chain_eq = [[f"v{i}", f"v{i + 1}"] for i in range(20)]
chain_val = [2.0] * 20
edge_cases.append((chain_eq, chain_val, [["v0", "v20"], ["v20", "v0"], ["v5", "v10"], ["v10", "v5"]],
                   [2.0 ** 20, 2.0 ** -20, 2.0 ** 5, 2.0 ** -5]))
for eq, val, qs, exp in edge_cases:
    for f in ALL:
        got = f(eq, val, qs)
        assert same_answers(got, exp), (f.__name__, got, exp)
    for sol in CLASSES:
        assert same_answers(sol.calcEquation(eq, val, qs), exp), type(sol).__name__
print("OK: biên (tự quy chiếu, chuỗi dài 20, tách nhóm, biến lạ) cho mọi cách")

OK: biên (tự quy chiếu, chuỗi dài 20, tách nhóm, biến lạ) cho mọi cách


In [23]:
# 12.3 Sinh ngẫu nhiên có oracle "giá trị ẩn"
NAMES = ["a", "b", "c", "d", "e", "f", "g", "h", "x1", "y2", "k9", "zz", "q"]


def gen_case(rng):
    n_vars = rng.randint(2, 9)
    names = rng.sample(NAMES, n_vars)
    hidden = {x: Fraction(rng.randint(1, 6), rng.randint(1, 3)) for x in names}   # giá trị ẩn dương
    n_eq = rng.randint(1, min(20, n_vars * 2))
    equations, values = [], []
    for _ in range(n_eq):
        a, b = rng.choice(names), rng.choice(names)
        r = hidden[a] / hidden[b]
        if not (0 < r <= 20):
            continue
        equations.append([a, b])
        values.append(float(r))
    if not equations:
        equations, values = [[names[0], names[0]]], [1.0]
    queries = []
    for _ in range(rng.randint(1, 20)):
        c = rng.choice(names + ["unk1", "unk2"])
        d = rng.choice(names + ["unk1", "unk2"])
        queries.append([c, d])
    return hidden, equations, values, queries


def oracle(hidden, equations, queries):
    # thành phần liên thông bằng gộp tập thô (không dùng Union–Find / đồ thị)
    groups = []
    for a, b in equations:
        merged = {a, b}
        rest = []
        for g in groups:
            if g & merged:
                merged |= g
            else:
                rest.append(g)
        groups = rest + [merged]
    res = []
    for c, d in queries:
        if any(c in g and d in g for g in groups):
            res.append(float(hidden[c] / hidden[d]))
        else:
            res.append(-1.0)
    return res


rng = random.Random(399)
for _ in range(2000):
    hidden, eq, val, qs = gen_case(rng)
    exp = oracle(hidden, eq, qs)
    for f in ALL:
        got = f(eq, val, qs)
        assert same_answers(got, exp), (f.__name__, eq, val, qs, got, exp)
    for sol in CLASSES:
        got = sol.calcEquation([e[:] for e in eq], val[:], [q[:] for q in qs])
        assert same_answers(got, exp), (type(sol).__name__, eq, val, qs, got, exp)
print("OK: 2000 input ngẫu nhiên (seed 399), 5 hàm + 5 lớp đều khớp oracle giá trị ẩn")

OK: 2000 input ngẫu nhiên (seed 399), 5 hàm + 5 lớp đều khớp oracle giá trị ẩn


In [24]:
# 12.4 Tính chất: nghịch đảo và nhân nối đuôi (với mọi cặp cùng nhóm): (c/d) * (d/c) = 1, (c/d)*(d/e) = c/e
rng = random.Random(400)
for _ in range(500):
    hidden, eq, val, qs = gen_case(rng)
    names = list(hidden)
    for _ in range(10):
        c, d, e = rng.choice(names), rng.choice(names), rng.choice(names)
        r = Solution().calcEquation(eq, val, [[c, d], [d, c], [d, e], [c, e]])
        if r[0] != -1.0:
            assert abs(r[0] * r[1] - 1.0) < 1e-6
            if r[2] != -1.0:
                assert abs(r[0] * r[2] - r[3]) < 1e-6 * max(1.0, abs(r[3]))
print("OK: tính chất nghịch đảo và nối đuôi")

OK: tính chất nghịch đảo và nối đuôi


In [25]:
# 12.5 Đầu vào không bị sửa đổi sau khi gọi, và kết quả lặp lại được (không có trạng thái ngầm)
eq, val, qs, exp = examples[0]
eq0, val0, qs0 = [e[:] for e in eq], val[:], [q[:] for q in qs]
r1 = Solution().calcEquation(eq, val, qs)
r2 = Solution().calcEquation(eq, val, qs)
assert eq == eq0 and val == val0 and qs == qs0
assert r1 == r2
print("OK: không đổi đầu vào, kết quả lặp lại")

OK: không đổi đầu vào, kết quả lặp lại


In [26]:
# 12.6 Hiệu năng: mở rộng ngoài ràng buộc để thấy khác biệt (V = 600 biến, 600 đẳng thức, 600 truy vấn)
rng = random.Random(600)
names = [f"n{i}" for i in range(600)]
hidden = {x: Fraction(rng.randint(1, 5), rng.randint(1, 3)) for x in names}
eq_big = [[names[i], names[i + 1]] for i in range(599)]
val_big = [float(hidden[a] / hidden[b]) for a, b in eq_big]
q_big = [[rng.choice(names), rng.choice(names)] for _ in range(600)]
exp_big = [float(hidden[c] / hidden[d]) for c, d in q_big]
for name, sol in [("UF", SolutionUF()), ("BFS", SolutionBFS())]:
    t = time.perf_counter()
    got = sol.calcEquation(eq_big, val_big, q_big)
    dt = time.perf_counter() - t
    assert same_answers(got, exp_big, tol=1e-6 * 1e6)       # chuỗi 600 cạnh: sai số tương đối nhỏ, cho dung sai rộng
    print(f"{name}: chạy xong dưới 5 giây?", dt < 5.0)
    assert dt < 5.0

UF: chạy xong dưới 5 giây? True
BFS: chạy xong dưới 5 giây? True


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên cạnh ngược `1 / v`

In [27]:
def wrong_no_reverse(equations, values, queries):
    graph = defaultdict(dict)
    for (a, b), v in zip(equations, values):
        graph[a][b] = v                              # SAI: thiếu graph[b][a] = 1 / v
        graph.setdefault(b, {})
    def dfs(u, t, acc, seen):
        if u == t:
            return acc
        seen.add(u)
        for nxt, w in graph[u].items():
            if nxt not in seen:
                r = dfs(nxt, t, acc * w, seen)
                if r != -1.0:
                    return r
        return -1.0
    return [dfs(c, d, 1.0, set()) if c in graph and d in graph else -1.0 for c, d in queries]


eq, val, qs, exp = examples[0]
print("sai:", wrong_no_reverse(eq, val, qs), "| đúng:", Solution().calcEquation(eq, val, qs))
assert not same_answers(wrong_no_reverse(eq, val, qs), exp)

sai: [6.0, -1.0, -1.0, 1.0, -1.0] | đúng: [6.0, 0.5, -1.0, 1.0, -1.0]


**Vì sao sai:** truy vấn `b / a` cần cạnh `b → a` trọng số `1/2`; không có cạnh ngược ta trả `-1.0` thay vì `0.5`. **Cách sửa:** với mỗi đẳng thức thêm **hai** cạnh `a → b: v` và `b → a: 1 / v`.

### Lỗi 2: không kiểm tra biến chưa xuất hiện (`x / x` phải là `-1.0`)

In [28]:
def wrong_same_is_one(equations, values, queries):
    ok = calc_dfs(equations, values, queries)
    return [1.0 if c == d else r for (c, d), r in zip(queries, ok)]           # SAI: coi c == d luôn là 1.0


eq, val, qs, exp = examples[0]
print("sai:", wrong_same_is_one(eq, val, qs), "| đúng:", Solution().calcEquation(eq, val, qs))
assert not same_answers(wrong_same_is_one(eq, val, qs), exp)

sai: [6.0, 0.5, -1.0, 1.0, 1.0] | đúng: [6.0, 0.5, -1.0, 1.0, -1.0]


**Vì sao sai:** đề nói biến không xuất hiện trong `equations` là **chưa xác định**, nên `x / x = -1.0` (truy vấn cuối của ví dụ 1). Chỉ khi `x` **có mặt** thì `x / x = 1.0`. **Cách sửa:** kiểm tra `c in graph and d in graph` **trước** khi xét `c == d`.

### Lỗi 3: quên tập `visited` ⇒ đệ quy vô hạn

In [29]:
def wrong_no_visited(equations, values, queries):
    graph = build_graph(equations, values)

    def dfs(u, t, acc):
        if u == t:
            return acc
        for nxt, w in graph[u].items():            # SAI: không có visited, a -> b -> a -> b ...
            r = dfs(nxt, t, acc * w)
            if r != -1.0:
                return r
        return -1.0

    return [dfs(c, d, 1.0) if c in graph and d in graph else -1.0 for c, d in queries]


try:
    wrong_no_visited([["a", "b"], ["c", "d"]], [2.0, 3.0], [["a", "c"]])       # a và c khác nhóm -> đi vòng mãi
    print("không lỗi?")
except RecursionError:
    print("RecursionError: đệ quy vô hạn vì thiếu visited")

RecursionError: đệ quy vô hạn vì thiếu visited


**Vì sao sai:** đồ thị hai chiều: từ `a` sang `b` rồi `b` lại quay về `a`,... không bao giờ dừng khi đích không nằm trong nhóm. **Cách sửa:** tập `visited` (hoặc `seen` trong BFS).

### Lỗi 4: công thức gộp trọng số Union–Find sai

In [30]:
def wrong_uf_formula(equations, values, queries):
    parent, weight = {}, {}

    def find(x):
        if parent[x] != x:
            origin = parent[x]
            parent[x] = find(origin)
            weight[x] *= weight[origin]
        return parent[x]

    for a, b in equations:
        for x in (a, b):
            parent.setdefault(x, x)
            weight.setdefault(x, 1.0)
    for (a, b), v in zip(equations, values):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            weight[ra] = v * weight[a] / weight[b]         # SAI: đảo vị trí weight[a], weight[b]
    return [weight[c] / weight[d] if c in parent and d in parent and find(c) == find(d) else -1.0
            for c, d in queries]


eq = [["a", "b"], ["c", "a"]]          # a/b = 2, c/a = 3  =>  c/b = 6
val = [2.0, 3.0]
qs = [["c", "b"]]
print("sai:", wrong_uf_formula(eq, val, qs), "| đúng:", Solution().calcEquation(eq, val, qs), "(= 3 * 2 = 6)")
assert not same_answers(wrong_uf_formula(eq, val, qs), [6.0])
assert same_answers(Solution().calcEquation(eq, val, qs), [6.0])

sai: [1.5] | đúng: [6.0] (= 3 * 2 = 6)


**Vì sao sai:** `ra/rb = v * weight[b] / weight[a]` (dùng `b` ở tử, `a` ở mẫu). Khi `a` và `b` đều chưa gộp, `weight = 1` nên **hai công thức trùng nhau** và ví dụ nhỏ không lộ lỗi; chỉ khi một trong hai đã nằm dưới một gốc với trọng số khác `1` thì mới sai. Hãy luôn thử một ví dụ có **ít nhất hai lần gộp**.

### Lỗi 5: cộng thay vì nhân

Nếu thấy mình viết `acc + w` thay cho `acc * w`, hãy kiểm tra lại: tỉ số **nối đuôi** thì **nhân**; cộng chỉ dùng khi bạn làm việc với **logarit** của tỉ số (xem mục 15). Kết quả `a / c` ở ví dụ 1 sẽ là `5` thay vì `6`.

In [31]:
acc_sai = 0.0
for w in (2.0, 3.0):
    acc_sai += w
acc_dung = 1.0
for w in (2.0, 3.0):
    acc_dung *= w
print("cộng (sai):", acc_sai, "| nhân (đúng):", acc_dung)
assert acc_sai == 5.0 and acc_dung == 6.0

cộng (sai): 5.0 | nhân (đúng): 6.0


## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu nhận dạng:** "các đại lượng liên hệ theo tỉ số/hiệu, hỏi quan hệ giữa hai đại lượng" ⇒ **đồ thị có trọng số** hoặc **Union–Find có trọng số**. Các bài cùng họ: "Satisfiability of Equality Equations" (990), "Is Graph Bipartite", "Accounts Merge", bài hiệu `x - y = c` (potential).
- **Tỉ số thì nhân, hiệu thì cộng:** cùng khung, chỉ đổi phép toán và phần tử trung hòa (`1` hay `0`).
- **Nên chọn cách nào?** ít biến + nhiều truy vấn ⇒ Floyd–Warshall hoặc Union–Find; ít truy vấn ⇒ DFS/BFS; thêm đẳng thức theo thời gian thực ⇒ Union–Find.
- **Đầu vào không mâu thuẫn** cho phép dùng bất kỳ đường đi nào; nếu đề cho phép mâu thuẫn, ta phải kiểm tra tính nhất quán (xem biến thể).
- **Số thực:** so sánh bằng sai số; không dùng `==`.
- **Kiểm thử:** sinh "giá trị ẩn" để có đáp án thật — mẹo xây oracle cho mọi bài về quan hệ giữa các biến.

## 15. Biến thể

1. **Phát hiện mâu thuẫn:** cho `a / b = 2`, `b / c = 3`, `a / c = 5` ⇒ mâu thuẫn (vì `a / c = 6`). Dùng Union–Find có trọng số: khi `ra == rb` kiểm tra `weight[a] / weight[b] ≈ v`.
2. **Đẳng thức đến theo thời gian thực:** Union–Find xử lý trực tiếp, không dựng lại đồ thị.
3. **Đáp án chính xác bằng phân số (`Fraction`)** thay cho số thực để loại bỏ sai số.
4. **Quan hệ hiệu `a - b = v`:** thay nhân bằng cộng, `1` bằng `0`; còn gọi là "potential".
5. **Logarit:** `log(a/b) = log a - log b`; đường đi nhân thành đường đi **cộng** ⇒ bài đường đi ngắn nhất cổ điển (Bellman–Ford, phát hiện chu trình âm ⇔ mâu thuẫn).

In [32]:
def is_consistent(equations, values, tol=1e-9):
    parent, weight = {}, {}

    def find(x):
        if parent[x] != x:
            origin = parent[x]
            parent[x] = find(origin)
            weight[x] *= weight[origin]
        return parent[x]

    for a, b in equations:
        for x in (a, b):
            parent.setdefault(x, x)
            weight.setdefault(x, 1.0)
    for (a, b), v in zip(equations, values):
        ra, rb = find(a), find(b)
        if ra == rb:
            if abs(weight[a] / weight[b] - v) > tol * max(1.0, abs(v)):
                return False
        else:
            parent[ra] = rb
            weight[ra] = v * weight[b] / weight[a]
    return True


print(is_consistent([["a", "b"], ["b", "c"], ["a", "c"]], [2.0, 3.0, 6.0]))     # True
print(is_consistent([["a", "b"], ["b", "c"], ["a", "c"]], [2.0, 3.0, 5.0]))     # False


def calc_exact(equations, values, queries):
    """Đáp án bằng Fraction (values cho dạng chuỗi/Fraction để tránh sai số)."""
    graph = defaultdict(dict)
    for (a, b), v in zip(equations, values):
        v = Fraction(v)
        graph[a][b] = v
        graph[b][a] = 1 / v
    out = []
    for c, d in queries:
        if c not in graph or d not in graph:
            out.append(None)
            continue
        seen, stack, res = {c}, [(c, Fraction(1))], None
        while stack:
            u, acc = stack.pop()
            if u == d:
                res = acc
                break
            for nxt, w in graph[u].items():
                if nxt not in seen:
                    seen.add(nxt)
                    stack.append((nxt, acc * w))
        out.append(res)
    return out


print(calc_exact([["a", "b"], ["b", "c"]], ["2", "3"], [["a", "c"], ["c", "a"], ["a", "z"]]))

True
False
[Fraction(6, 1), Fraction(1, 6), None]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Bao đóng | lặp nối đuôi tới khi hết sinh thêm | `~O(V^4)` | `O(V^2)` | hiểu đề, `V` rất nhỏ |
| 2. DFS | tích dọc theo một đường đi, đệ quy | `O(Q(V+E))` | `O(V+E)` | ít truy vấn, code ngắn |
| 3. BFS | như DFS nhưng dùng hàng đợi | `O(Q(V+E))` | `O(V+E)` | tránh đệ quy sâu |
| 4. Floyd–Warshall | tính trước mọi cặp | `O(V^3 + Q)` | `O(V^2)` | `V` nhỏ, nhiều truy vấn |
| 5. Union–Find có trọng số | mỗi biến nhớ tỉ số so với gốc | `O((E+Q)α(V))` | `O(V)` | **nộp bài**; thêm đẳng thức động |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: (a) `1 / 0.5`; (b) với `a / b = 2`, `b / c = 3` thì `c / a` bằng bao nhiêu; (c) `graph = {"a": {"b": 2}}`, `graph["a"]["b"]` và `"b" in graph`; (d) `abs(0.1 * 3 - 0.3) < 1e-9`.

**Bài 2 (tính tay).** Với `x / y = 4`, `y / z = 0.5`, `w / z = 3`: tính `x / z`, `w / x`, `y / w` và `x / q` (với `q` lạ). Dựng cả đồ thị.

**Bài 3 (điền TODO).** Hoàn thành `todo_calc(equations, values, queries)` theo Cách 2 (DFS).

In [33]:
def todo_calc(equations, values, queries):
    # TODO 1: dựng graph (defaultdict(dict)): graph[a][b] = v và graph[b][a] = 1 / v
    # TODO 2: với mỗi truy vấn (c, d): nếu c hoặc d không có trong graph -> -1.0
    # TODO 3: viết hàm dfs(u, target, acc, visited): u == target thì trả acc; duyệt các láng giềng chưa thăm
    # TODO 4: gom kết quả vào danh sách và trả về
    return None

# KHÔNG gọi hàm TODO ở đây để Run All không lỗi (TODO chưa điền).

**Bài 4 (tự viết lại).** Viết `calc_floyd2(equations, values, queries)` theo Cách 4 (Floyd–Warshall) **không nhìn lại** mục 7; kiểm tra với ba ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** Viết `find_conflict(equations, values)` trả về **chỉ số đẳng thức đầu tiên gây mâu thuẫn** (hoặc `-1` nếu không có), xử lý các đẳng thức **theo thứ tự**. Kiểm tra: `[["a","b"],["b","c"],["a","c"]]` với `[2, 3, 5]` → `2`; với `[2, 3, 6]` → `-1`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `2.0`; `1/6`; `2` và `False`; `True`. Bài 3: nhìn lại `calc_dfs` ở mục 7.
- **Mức 2 (vừa):** Bài 2: `x/z = (x/y)(y/z) = 4*0.5 = 2`; `w/x = (w/z)/(x/z) = 3/2 = 1.5`; `y/w = (y/z)/(w/z) = 0.5/3 = 1/6`; `x/q = -1.0`. Bài 4: đánh số biến bằng `dict`, nhớ **vòng `k` ngoài cùng**.
- **Mức 3 (sâu):** Bài 5: dùng Union–Find có trọng số như `is_consistent` nhưng trả về chỉ số `i` ngay khi `ra == rb` mà `weight[a]/weight[b]` không xấp xỉ `v`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1
assert 1 / 0.5 == 2.0
r1 = Solution().calcEquation([["a", "b"], ["b", "c"]], [2.0, 3.0], [["c", "a"]])
assert abs(r1[0] - 1 / 6) < 1e-9
g1 = {"a": {"b": 2}}
assert g1["a"]["b"] == 2 and ("b" in g1) is False
assert abs(0.1 * 3 - 0.3) < 1e-9

# Bài 2
eq2, val2 = [["x", "y"], ["y", "z"], ["w", "z"]], [4.0, 0.5, 3.0]
r2 = Solution().calcEquation(eq2, val2, [["x", "z"], ["w", "x"], ["y", "w"], ["x", "q"]])
assert same_answers(r2, [2.0, 1.5, 1 / 6, -1.0])


# Bài 3
def todo_calc_ans(equations, values, queries):
    graph = defaultdict(dict)
    for (a, b), v in zip(equations, values):
        graph[a][b] = v
        graph[b][a] = 1 / v

    def dfs(u, target, acc, visited):
        if u == target:
            return acc
        visited.add(u)
        for nxt, w in graph[u].items():
            if nxt not in visited:
                res = dfs(nxt, target, acc * w, visited)
                if res != -1.0:
                    return res
        return -1.0

    return [dfs(c, d, 1.0, set()) if c in graph and d in graph else -1.0 for c, d in queries]


for eq, val, qs, exp in examples:
    assert same_answers(todo_calc_ans(eq, val, qs), exp)


# Bài 4
def calc_floyd2(equations, values, queries):
    ids = {}
    for a, b in equations:
        ids.setdefault(a, len(ids))
        ids.setdefault(b, len(ids))
    m = len(ids)
    g = [[-1.0] * m for _ in range(m)]
    for i in range(m):
        g[i][i] = 1.0
    for (a, b), v in zip(equations, values):
        g[ids[a]][ids[b]] = v
        g[ids[b]][ids[a]] = 1 / v
    for k in range(m):
        for i in range(m):
            for j in range(m):
                if g[i][j] < 0 and g[i][k] > 0 and g[k][j] > 0:
                    g[i][j] = g[i][k] * g[k][j]
    return [g[ids[c]][ids[d]] if c in ids and d in ids else -1.0 for c, d in queries]


for eq, val, qs, exp in examples:
    assert same_answers(calc_floyd2(eq, val, qs), exp)


# Bài 5
def find_conflict(equations, values, tol=1e-9):
    parent, weight = {}, {}

    def find(x):
        if parent[x] != x:
            origin = parent[x]
            parent[x] = find(origin)
            weight[x] *= weight[origin]
        return parent[x]

    for a, b in equations:
        for x in (a, b):
            parent.setdefault(x, x)
            weight.setdefault(x, 1.0)
    for i, ((a, b), v) in enumerate(zip(equations, values)):
        ra, rb = find(a), find(b)
        if ra == rb:
            if abs(weight[a] / weight[b] - v) > tol * max(1.0, abs(v)):
                return i
        else:
            parent[ra] = rb
            weight[ra] = v * weight[b] / weight[a]
    return -1


assert find_conflict([["a", "b"], ["b", "c"], ["a", "c"]], [2, 3, 5]) == 2
assert find_conflict([["a", "b"], ["b", "c"], ["a", "c"]], [2, 3, 6]) == -1
print("OK: toàn bộ đáp án đúng")

OK: toàn bộ đáp án đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template** (Union–Find có trọng số)

```python
class Solution:
    def calcEquation(self, equations: List[List[str]], values: List[float], queries: List[List[str]]) -> List[float]:
        parent, weight = {}, {}          # weight[x] = x / parent[x]

        def find(x):
            if parent[x] != x:
                origin = parent[x]
                parent[x] = find(origin)
                weight[x] *= weight[origin]
            return parent[x]

        for a, b in equations:
            for x in (a, b):
                if x not in parent:
                    parent[x], weight[x] = x, 1.0
        for (a, b), v in zip(equations, values):
            ra, rb = find(a), find(b)
            if ra != rb:
                parent[ra] = rb
                weight[ra] = v * weight[b] / weight[a]
        return [weight[c] / weight[d] if c in parent and d in parent and find(c) == find(d) else -1.0
                for c, d in queries]
```

**Checklist**

- [ ] Mỗi đẳng thức `a / b = v` tạo **cả hai** chiều (`v` và `1 / v`) nếu dùng đồ thị?
- [ ] Biến **chưa xuất hiện** (kể cả `x / x`) trả `-1.0`?
- [ ] DFS/BFS có `visited`/`seen`?
- [ ] Union–Find: sau `find`, `weight[x]` đã nhân dồn đúng; công thức gộp `weight[ra] = v * weight[b] / weight[a]`?
- [ ] Floyd–Warshall: vòng ngoài cùng là đỉnh trung gian `k`; dùng **nhân**, không cộng?
- [ ] So số thực bằng sai số `1e-5`; đã thử chuỗi dài, nhóm rời nhau, tự quy chiếu `a / a`?